# 项目 — 航空公司 AI 助手

## 练习目标

把第 2 周学到的内容整合起来，为一家航空公司打造 **AI 客户支持助手**：

- 用 **Gradio** 做聊天界面
- 用 **Chat Completions** 接 OpenAI（或本地 Ollama）
- 学习 **Tools（工具调用）**：让模型在对话中请求你本地的函数
- 进一步：用 **SQLite** 存票价，并支持「查价 / 改价」两个工具

## 和本课 Day 4 的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|------------------|
| system prompt | `system_message` 约束助手语气与准确性 |
| Gradio ChatInterface | `gr.ChatInterface(fn=chat, type="messages")` |
| Tools / function calling | `tools=` + `finish_reason=="tool_calls"` |
| 多工具 / 循环调用 | `handle_tool_calls` + `while` |
| 外部数据源 | 字典 → SQLite `prices.db` |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY=...`
2. 从上到下依次运行单元格（Shift+Enter）
3. 在 Gradio 聊天框里试：「How much is a ticket to London?」


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：解析工具调用参数（LLM 常把 arguments 以 JSON 字符串返回）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI
# 导入 gradio：快速搭聊天 UI（ChatInterface / Blocks）
import gradio as gr


In [ ]:
# ========== 初始化：读密钥、选模型、创建客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量取出 OpenAI API Key（常见名字是 OPENAI_API_KEY）
openai_api_key = os.getenv('OPENAI_API_KEY')
# 有密钥就打印前 8 位做确认（不要打印完整密钥）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    # 没配置时给出提示，后续调用会失败
    print("OpenAI API Key not set")

# 模型 id：集中写在常量里，后面 chat.completions.create 都用它
MODEL = "gpt-4.1-mini"
# 创建 OpenAI 客户端；默认会自动读环境变量 OPENAI_API_KEY
openai = OpenAI()

# 备选：若想改用本地 Ollama（OpenAI 兼容 /v1），取消下面两行注释
# 先确认本机 Ollama 已启动（参见 week1/day2）
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
# ========== 系统提示词（System Prompt）：定角色与回答风格 ==========
# 注意：发给模型的英文内容保持原样，不要翻译——它会影响模型行为

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
# ========== 第一版 chat：无工具，纯多轮对话 ==========

# 收集每次 API 原始响应，方便后面单元格拆开查看结构
response_list = []

def chat(message, history):
    # Gradio type="messages" 时 history 是 dict 列表；这里再规范成 role/content
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # messages = system + 历史 + 当前用户句（Chat Completions 标准格式）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用云端模型；本阶段尚未传入 tools=
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    # 存下完整 response 对象，供调试单元格打印
    response_list.append(response)
    # 取出助手文本回复给 Gradio 显示
    return response.choices[0].message.content

# 启动 Gradio 聊天界面；type="messages" 使用 OpenAI 风格消息列表
gr.ChatInterface(fn=chat, type="messages").launch()


In [ ]:
# 查看刚才累计的原始响应对象列表（ChatCompletion 实例）
response_list


In [ ]:
# ========== 拆开看响应结构：choices / message / finish_reason ==========
# 后面做工具调用时，常会检查 finish_reason=="tool_calls"
# 以及 message.content（有工具调用时 content 可能为空）

print(response_list[0])
print(response_list[0].choices)
print(response_list[0].choices[0])


## 工具（Tools）

工具是前沿 LLM 提供的一项极其强大的功能。

有了工具，你可以编写一个函数，并让 LLM 在响应过程中**请求调用**该函数。

听起来几乎有点诡异……我们是在给它在我们机器上运行代码的权力？

嗯，有那么一点——但真正执行的是**你自己写的 Python**；模型只是用约定好的 JSON 告诉你「想调哪个函数、参数是什么」。


In [ ]:
# ========== 先写一个「真有用」的本地函数：查票价 ==========

# 城市名（小写）→ 票价字符串；模拟外部数据源
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    # 打印日志：确认工具真的被调用了（调试很重要）
    print(f"Tool called for city {destination_city}")
    # .lower() 做大小写不敏感查找；找不到就返回 Unknown...
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    # 返回给模型看的自然语言结果（会作为 tool 消息的 content）
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
# 本地直接测一下函数：不经过 LLM，确认字典查找正常
get_ticket_price("London")


In [ ]:
# ========== 工具描述（JSON Schema）：告诉模型「有什么函数可调」 ==========
# OpenAI function calling 要求用特定字典结构描述 name / description / parameters

price_function = {
    # 函数名：必须和后面 handle_tool_call 里分支判断的名字一致
    "name": "get_ticket_price",
    # 给模型看的说明：模型靠这段文字决定何时调用
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                # 参数含义说明：帮助模型正确填参
                "description": "The city that the customer wants to travel to",
            },
        },
        # 必填参数列表
        "required": ["destination_city"],
        # 不允许额外未知字段
        "additionalProperties": False
    }
}


In [ ]:
# ========== 组装 tools 列表：API 期望 type=function + function=schema ==========

tools = [{"type": "function", "function": price_function}]


In [ ]:
# 打印 tools，确认结构是否正确（应是含一个 function 的列表）
tools


## 让 OpenAI 使用我们的工具

要让 OpenAI「调用我们的工具」，有一些琐碎细节。

我们实际做的是：给 LLM 机会**告知我们**它希望我们运行该工具（`finish_reason=="tool_calls"`），然后由**我们的代码**执行函数，再把结果塞回 `messages`，请模型生成最终自然语言回答。

新的 `chat` 函数大致如下：


In [ ]:
# ========== 带工具的 chat：一次 tool_calls → 本地执行 → 再问模型 ==========

def chat(message, history):
    # 规范化 Gradio history 为 role/content 字典列表
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼完整 messages：system + 历史 + 当前用户
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调试：看真正发给模型的消息
    print("Messages sent to model = ", messages)
    # 关键：传入 tools=，模型才可能返回 tool_calls
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    # 若模型决定调用工具（而不是直接文本回答）
    if response.choices[0].finish_reason=="tool_calls":
        # 取出带 tool_calls 的 assistant message（需原样 append 回 messages）
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        # 本地执行工具，得到 role=tool 的结果消息
        response = handle_tool_call(message)
        print("Tool response = ", response)
        # 把「助手的工具请求」和「工具结果」都追加进对话上下文
        messages.append(message)
        messages.append(response)
        print("New messages list = ", messages)
        # 第二次调用：模型根据工具结果生成最终用户可见回复
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        print("New response after tool call = ", response)

    # 返回最终文本
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_call：把模型的 tool_calls 变成真实函数执行 ==========

def handle_tool_call(message):
    # 本版本只处理第一个 tool_call（后面会扩展为多个）
    tool_call = message.tool_calls[0]
    # 按函数名分发；这里只实现 get_ticket_price
    if tool_call.function.name == "get_ticket_price":
        # arguments 是 JSON 字符串 → 解析成 Python dict
        arguments = json.loads(tool_call.function.arguments)
        # 取出城市参数
        city = arguments.get('destination_city')
        # 调用真正的本地函数
        price_details = get_ticket_price(city)
        # 组装 role=tool 的消息；tool_call_id 必须对应请求里的 id
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


In [ ]:
# 启动带工具调用的聊天界面；试问某城市票价，观察终端 Tool called... 日志
gr.ChatInterface(fn=chat, type="messages").launch()


## 让我们做几项改进

1. **在一次响应中处理多个工具调用**：模型可能在同一条消息里返回多个 `tool_calls`
2. **一个接一个地处理多轮工具调用**：第一轮工具结果回来后，模型可能还要再调工具（后面用 `while`）


In [ ]:
# ========== 改进版 chat：一次响应里可能有多个 tool_calls（用 extend） ==========

def chat(message, history):
    # 规范化历史消息
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼 messages
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    # 带 tools 的第一次调用
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        # handle_tool_calls（复数）：返回多条 tool 结果列表
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        # 先 append 助手消息，再 extend 多条 tool 结果
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        # 第二次生成最终回复（本版第二次调用未再传 tools）
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        print("New response after tool call = ", response)

    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_calls：遍历 message.tool_calls，逐个执行 ==========

def handle_tool_calls(message):
    # 收集所有 role=tool 的结果
    responses = []
    for tool_call in message.tool_calls:
        # 目前仍只识别 get_ticket_price
        if tool_call.function.name == "get_ticket_price":
            # 解析 JSON 参数
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            # 每条结果都要带对应的 tool_call_id
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# 启动支持「一次多工具结果」的聊天界面
gr.ChatInterface(fn=chat, type="messages").launch()


In [ ]:
# ========== 再改进：while 循环，允许多轮连续 tool_calls ==========
# 第二次及以后的 create 也传 tools=，模型才能继续请求工具

def chat(message, history):
    # 规范化历史
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼 messages
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    # 只要还在要工具，就继续：执行 → 追加 → 再问模型
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        # 注意：这里继续传 tools=，支持链式多轮工具
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print("New response after tool call = ", response)

    return response.choices[0].message.content


In [ ]:
# ========== 引入 SQLite：把票价从内存字典换成持久化数据库 ==========

import sqlite3


In [ ]:
# ========== 建库建表：prices(city PRIMARY KEY, price REAL) ==========

# 数据库文件名（相对当前工作目录）
DB = "prices.db"

# with 连接：退出块时自动关闭；CREATE IF NOT EXISTS 可重复运行
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # 城市主键 + 价格浮点数
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()


In [ ]:
# ========== 重写 get_ticket_price：从 SQLite 查询 ==========

def get_ticket_price(city):
    # flush=True：日志立刻打出，方便在 Gradio 场景观察
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询（? 占位），避免拼接 SQL；城市统一小写
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        # 有行就格式化返回；否则提示无数据
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [ ]:
# 查询测试：若库里还没写入 London，可能得到无数据提示
get_ticket_price("London")


In [ ]:
# ========== set_ticket_price：写入/更新票价（UPSERT） ==========

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # INSERT ... ON CONFLICT DO UPDATE：有则改价，无则插入
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()


In [ ]:
# ========== 种子数据：把若干城市票价写入数据库 ==========

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
# 逐条调用 set_ticket_price 写入
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [ ]:
# 写入后再查 Tokyo，应能看到数据库里的价格
get_ticket_price("Tokyo")


In [ ]:
# 用当前 while 版 chat + SQLite 版 get_ticket_price 启动界面
gr.ChatInterface(fn=chat, type="messages").launch()


## 练习

添加一个用于**设置机票价格**的工具！

提示：需要（1）`set_ticket_price` 的函数 schema；（2）把它放进 `tools`；（3）在 `handle_tool_calls` 里增加对应分支。


In [ ]:
# ========== 练习答案骨架：同时注册「查价」和「改价」两个工具 ==========

# 查价工具的 JSON Schema（给模型看）
price_get_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# 改价工具的 JSON Schema：多一个 number 类型的 price 参数
price_set_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The price of the ticket",
            }
        },
        "required": ["destination_city","price"],
        "additionalProperties": False
    }
}

# tools 列表里放两个 function
tools = [{"type": "function", "function": price_get_function}, {"type": "function", "function": price_set_function}]


In [ ]:
# ========== 双工具版 chat：while + tools=（与前面多轮逻辑相同） ==========

def chat(message, history):
    # 规范化历史
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼 messages
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        # 现在 handle_tool_calls 能处理 get / set 两种
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print("New response after tool call = ", response)

    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_calls：分发 get_ticket_price / set_ticket_price ==========

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        # 分支 A：查价
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        # 分支 B：改价（写入 SQLite）
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            # 真正写库
            set_ticket_price(city, price)
            # 把执行结果用自然语言回传给模型
            responses.append({
                "role": "tool",
                "content": f"Price for {city} set to {price}",
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# 最终演示：试「查 London 票价」和「把 Paris 改成某个价格」
gr.ChatInterface(fn=chat, type="messages").launch()


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">这几乎不必多说！你现在已能给 LLM 赋予行动能力。这个航空助手不再只会回答问题——它还可以与预订 API 交互来完成预订！</span>
        </td>
    </tr>
</table>
